In [ ]:
"""
analyzer_opencv.py  —  Method 1: Pure OpenCV Analysis
------------------------------------------------------
Zero cost, zero API key, works offline.
Uses computer vision techniques:
  - Motion detection between frames (frame differencing)
  - Color analysis (dominant scene color, brightness)
  - Contour / blob detection (counts moving regions)
  - Brightness-based scene classification

No ML model needed. Fast and lightweight.
"""

import cv2
import numpy as np
from pathlib import Path


# ─── Scene Rules ─────────────────────────────────────────────────────────────
# You can customize these thresholds for your footage

MOTION_THRESHOLD   = 1500   # pixel area: contours larger than this count as motion
BRIGHTNESS_DARK    = 60     # mean pixel value → "dark / night scene"
BRIGHTNESS_BRIGHT  = 180    # mean pixel value → "overexposed / outdoor"
BLUE_DOMINANCE     = 30     # blue channel excess over R/G → "blue vehicle likely"
RED_DOMINANCE      = 30     # red  channel excess → "red vehicle"


def analyze_frame_opencv(frame_path: str, prev_frame_path: str = None) -> dict:
    """
    Analyze a single frame with pure OpenCV.
    Optionally compare with previous frame for motion detection.
    """
    img = cv2.imread(frame_path)
    if img is None:
        return {"error": f"Could not read {frame_path}"}

    h, w = img.shape[:2]
    result = {}

    # ── 1. Brightness & exposure ──────────────────────────────────────────
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    brightness = float(np.mean(gray))
    result["brightness"] = round(brightness, 1)

    if brightness < BRIGHTNESS_DARK:
        result["lighting"] = "night / low-light"
    elif brightness > BRIGHTNESS_BRIGHT:
        result["lighting"] = "bright / outdoor"
    else:
        result["lighting"] = "normal / indoor"

    # ── 2. Dominant color analysis ────────────────────────────────────────
    b_mean = float(np.mean(img[:, :, 0]))
    g_mean = float(np.mean(img[:, :, 1]))
    r_mean = float(np.mean(img[:, :, 2]))

    dominant_colors = []
    if b_mean > r_mean + BLUE_DOMINANCE and b_mean > g_mean + BLUE_DOMINANCE:
        dominant_colors.append("blue")
    if r_mean > b_mean + RED_DOMINANCE and r_mean > g_mean + RED_DOMINANCE:
        dominant_colors.append("red")
    if g_mean > r_mean + 20 and g_mean > b_mean + 20:
        dominant_colors.append("green")
    if not dominant_colors:
        dominant_colors.append("neutral/gray")

    result["dominant_colors"] = dominant_colors

    # ── 3. Edge density (scene complexity) ───────────────────────────────
    edges = cv2.Canny(gray, 50, 150)
    edge_density = float(np.mean(edges > 0))
    result["edge_density"] = round(edge_density, 3)

    if edge_density > 0.15:
        result["scene_complexity"] = "high (many objects / busy scene)"
    elif edge_density > 0.07:
        result["scene_complexity"] = "medium"
    else:
        result["scene_complexity"] = "low (empty / simple scene)"

    # ── 4. Motion detection (frame differencing) ─────────────────────────
    motion_detected = False
    motion_regions  = 0
    motion_area     = 0

    if prev_frame_path:
        prev = cv2.imread(prev_frame_path)
        if prev is not None:
            prev_g = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
            diff   = cv2.absdiff(gray, prev_g)
            _, thresh = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)
            # Morphological cleanup
            kernel  = np.ones((5, 5), np.uint8)
            cleaned = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
            contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            big = [c for c in contours if cv2.contourArea(c) > MOTION_THRESHOLD]
            motion_regions = len(big)
            motion_area    = sum(cv2.contourArea(c) for c in big)
            motion_detected = motion_regions > 0

    result["motion_detected"] = motion_detected
    result["motion_regions"]  = motion_regions
    result["motion_area_px"]  = round(motion_area)

    # ── 5. Large-blob detection (rough object count) ──────────────────────
    _, bin_img = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(bin_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    large_blobs = [c for c in contours if cv2.contourArea(c) > (h * w * 0.02)]
    result["large_object_count"] = len(large_blobs)

    # ── 6. Generate human-readable description ────────────────────────────
    parts = []
    parts.append(f"{result['lighting'].capitalize()} scene")

    if motion_detected:
        parts.append(f"motion detected in {motion_regions} region(s)")

    if dominant_colors != ["neutral/gray"]:
        parts.append(f"dominant color: {', '.join(dominant_colors)}")

    parts.append(result["scene_complexity"])

    if large_blobs:
        parts.append(f"{len(large_blobs)} large object(s) detected")

    result["description"] = ". ".join(parts) + "."

    # ── 7. Tags & alert ──────────────────────────────────────────────────
    tags = []
    if motion_detected:
        tags.append("motion")
    if "blue" in dominant_colors or "red" in dominant_colors:
        tags.append("colored-object")
    if edge_density > 0.12:
        tags.append("busy-scene")

    alert = motion_detected and motion_area > 8000
    result["tags"]  = tags
    result["alert"] = alert
    result["alert_reason"] = (
        f"Large motion area ({result['motion_area_px']}px) across {motion_regions} region(s)"
        if alert else None
    )
    result["confidence"] = "medium"
    result["method"]     = "opencv"

    return result


def analyze_all_opencv(frames: list[dict]) -> list[dict]:
    results = []
    for i, frame in enumerate(frames):
        prev_path = frames[i - 1]["file_path"] if i > 0 else None
        print(f"  Frame {frame['frame_num']:>3}/{len(frames)}  [{frame['timestamp']}]", end="  ")
        analysis = analyze_frame_opencv(frame["file_path"], prev_path)
        merged = {**frame, **analysis}
        status = "🚨 ALERT" if analysis.get("alert") else "✓"
        print(f"{status}  {analysis.get('description','')[:65]}")
        results.append(merged)
    return results

In [1]:
import json
import datetime
from dataclasses import dataclass, field
from typing import List, Optional


# ── Data models ──────────────────────────────────────────

@dataclass
class TelemetryFrame:
    frame_id: int
    timestamp: str
    location: str
    altitude_m: float
    description: str          # simulated frame text


@dataclass
class ProcessedFrame:
    frame_id: int
    timestamp: str
    location: str
    raw_description: str
    vlm_caption: str
    detected_objects: List[str]
    alert: Optional[str] = None


# ── Mock VLM (swap for real BLIP-2 in production) ────────

class MockVLM:
    """Simulates BLIP-2 output from text frame descriptions."""

    OBJECT_KEYWORDS = {
        "truck": "vehicle", "car": "vehicle", "van": "vehicle",
        "f150": "vehicle", "ford": "vehicle",
        "person": "person", "man": "person", "woman": "person",
        "gate": "landmark", "garage": "landmark",
        "entrance": "landmark",
    }

    def caption(self, description: str) -> str:
        # In real mode this calls BLIP-2; here we enrich the raw text
        desc = description.lower()
        enrichments = []
        if "midnight" in desc or "00:" in desc:
            enrichments.append("captured at night")
        if "gate" in desc:
            enrichments.append("near perimeter gate")
        suffix = ", ".join(enrichments)
        return f"{description.strip()}{', ' + suffix if suffix else ''}."

    def detect_objects(self, description: str) -> List[str]:
        found = set()
        for kw, category in self.OBJECT_KEYWORDS.items():
            if kw in description.lower():
                found.add(f"{kw} ({category})")
        return sorted(found)


# ── Alert rules ──────────────────────────────────────────

def evaluate_alerts(frame: TelemetryFrame, caption: str) -> Optional[str]:
    desc = caption.lower()
    hour = int(frame.timestamp.split(":")[0])

    if "person" in desc and (hour >= 22 or hour <= 5):
        return f"ALERT: Person detected at {frame.location} at {frame.timestamp}"
    if "loiter" in desc:
        return f"ALERT: Loitering detected at {frame.location}"
    if "unknown" in desc and "vehicle" in desc:
        return f"ALERT: Unknown vehicle at {frame.location}"
    return None


# ── Pipeline ─────────────────────────────────────────────

def process_frames(frames: List[TelemetryFrame]) -> List[ProcessedFrame]:
    vlm = MockVLM()
    results = []
    for f in frames:
        caption = vlm.caption(f.description)
        objects = vlm.detect_objects(f.description)
        alert = evaluate_alerts(f, caption)
        results.append(ProcessedFrame(
            frame_id=f.frame_id,
            timestamp=f.timestamp,
            location=f.location,
            raw_description=f.description,
            vlm_caption=caption,
            detected_objects=objects,
            alert=alert,
        ))
        if alert:
            print(f"[!] {alert}")
    return results


# ── Simulated input data ──────────────────────────────────

SIMULATED_FRAMES = [
    TelemetryFrame(1,  "12:00", "garage",    10.0, "Blue Ford F150 parked at garage"),
    TelemetryFrame(2,  "14:30", "main gate", 12.0, "Delivery van entering main gate"),
    TelemetryFrame(3,  "00:01", "main gate", 8.0,  "Person loitering near main gate"),
    TelemetryFrame(4,  "00:15", "entrance",  9.0,  "Unknown vehicle at side entrance"),
    TelemetryFrame(5,  "08:00", "garage",    10.0, "Blue Ford F150 at garage again"),
]

if __name__ == "__main__":
    processed = process_frames(SIMULATED_FRAMES)
    for p in processed:
        print(json.dumps(p.__dict__, indent=2))

[!] ALERT: Person detected at main gate at 00:01
[!] ALERT: Unknown vehicle at entrance
{
  "frame_id": 1,
  "timestamp": "12:00",
  "location": "garage",
  "raw_description": "Blue Ford F150 parked at garage",
  "vlm_caption": "Blue Ford F150 parked at garage.",
  "detected_objects": [
    "f150 (vehicle)",
    "ford (vehicle)",
    "garage (landmark)"
  ],
  "alert": null
}
{
  "frame_id": 2,
  "timestamp": "14:30",
  "location": "main gate",
  "raw_description": "Delivery van entering main gate",
  "vlm_caption": "Delivery van entering main gate, near perimeter gate.",
  "detected_objects": [
    "gate (landmark)",
    "van (vehicle)"
  ],
  "alert": null
}
{
  "frame_id": 3,
  "timestamp": "00:01",
  "location": "main gate",
  "raw_description": "Person loitering near main gate",
  "vlm_caption": "Person loitering near main gate, near perimeter gate.",
  "detected_objects": [
    "gate (landmark)",
    "person (person)"
  ],
  "alert": "ALERT: Person detected at main gate at 00:01"

In [2]:
%pip install transformers accelerate Pillow torch torchvision

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from PIL import Image
import torch
import requests
from io import BytesIO


class RealBLIP2VLM:
    """Real BLIP-2 — same interface as MockVLM, drop-in swap."""

    MODEL_ID = "Salesforce/blip2-opt-2.7b"  # ~5.5 GB download

    def __init__(self):
        print("Loading BLIP-2 (first run downloads ~5.5 GB)...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.processor = Blip2Processor.from_pretrained(self.MODEL_ID)
        self.model = Blip2ForConditionalGeneration.from_pretrained(
            self.MODEL_ID,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto",
        )
        print(f"BLIP-2 loaded on {self.device}")

    def _load_image(self, image_source: str) -> Image.Image:
        # Accepts file path or URL
        if image_source.startswith("http"):
            resp = requests.get(image_source, timeout=10)
            return Image.open(BytesIO(resp.content)).convert("RGB")
        return Image.open(image_source).convert("RGB")

    def caption(self, image_source: str) -> str:
        image = self._load_image(image_source)
        inputs = self.processor(images=image, return_tensors="pt").to(self.device)
        output = self.model.generate(**inputs, max_new_tokens=50)
        return self.processor.decode(output[0], skip_special_tokens=True)

    def answer(self, image_source: str, question: str) -> str:
        # Visual Q&A — used for bonus follow-up feature
        image = self._load_image(image_source)
        inputs = self.processor(
            images=image,
            text=question,
            return_tensors="pt"
        ).to(self.device)
        output = self.model.generate(**inputs, max_new_tokens=30)
        return self.processor.decode(output[0], skip_special_tokens=True)

    def detect_objects(self, image_source: str) -> List[str]:
        # Ask BLIP-2 directly what objects are present
        answer = self.answer(image_source, "What objects are visible in this image?")
        return [answer]  # parse further as needed

vl=RealBLIP2VLM()
print(vl.detect_objects(r"C:\Users\Samarth Patil\Pictures\Screenshots\Screenshot 2026-01-09 200916.png"))

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading BLIP-2 (first run downloads ~5.5 GB)...


preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

c:\Users\Samarth Patil\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Samarth Patil\.cache\huggingface\hub\models--Salesforce--blip2-opt-2.7b. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

... (8 outputs left out)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00002-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Error while downloading from https://huggingface.co/Salesforce/blip2-opt-2.7b/resolve/59a1ef6c1e5117b3f65523d1c6066825bcf315e3/model-00002-of-00002.safetensors: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...


OSError: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.